In [51]:
import GtoTmodel as GtoTmodel
import Circuits as Circuits 
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np

In [52]:
circuits = Circuits.CircuitS()


Graph 4['VDD', 'VSS', 'VIN1', 'NM1', 'NM1_D', 'NM1_G', 'NM1_S', 'NM1_B']
Graph 6['VDD', 'VSS', 'VOUT1', 'NM1', 'NM1_D', 'NM1_G', 'NM1_S', 'NM1_B', 'R1', 'R1_P', 'R1_N', 'C1', 'C1_P', 'C1_N']
Graph 9['VDD', 'VSS', 'VIN1', 'VOUT1', 'VB1', 'PM1', 'PM1_D', 'PM1_G', 'PM1_S', 'PM1_B', 'NM1', 'NM1_D', 'NM1_G', 'NM1_S', 'NM1_B']
Graph 14['VDD', 'VSS', 'VIN1', 'VOUT1', 'NM1', 'NM1_D', 'NM1_G', 'NM1_S', 'NM1_B', 'PM1', 'PM1_D', 'PM1_G', 'PM1_S', 'PM1_B', 'R1', 'R1_P', 'R1_N']
Graph 17['VDD', 'VSS', 'VIN1', 'VOUT1', 'R1', 'R1_P', 'R1_N', 'R2', 'R2_P', 'R2_N', 'NM1', 'NM1_D', 'NM1_G', 'NM1_S', 'NM1_B', 'NM2', 'NM2_D', 'NM2_G', 'NM2_S', 'NM2_B']
Graph 20['VDD', 'VSS', 'VIN1', 'VOUT1', 'VB1', 'NM1', 'NM1_D', 'NM1_G', 'NM1_S', 'NM1_B', 'NM2', 'NM2_D', 'NM2_G', 'NM2_S', 'NM2_B', 'R1', 'R1_P', 'R1_N']
Graph 22['VDD', 'VSS', 'VIN1', 'VOUT1', 'VB1', 'PM1', 'PM1_D', 'PM1_G', 'PM1_S', 'PM1_B', 'PM2', 'PM2_D', 'PM2_G', 'PM2_S', 'PM2_B']
Graph 24['VDD', 'VSS', 'VIN1', 'VOUT1', 'NM1', 'NM1_D', 'NM1_G', 'NM1_

In [53]:
# Assuming circuits has a method or attribute to get the matrix, e.g., circuits.get_matrix()
matrix = circuits.component_lists  # Replace with the actual method or attribute
max_length = max(len(vector) for vector in matrix)
print("Maximum length of vectors in the matrix:", max_length)

Maximum length of vectors in the matrix: 310


In [54]:
circuits.vocab.__len__()  # This should give the number of components

892

In [55]:
torch.manual_seed(1337)
torch.cuda.manual_seed(1337)
embed_dim = 16  # Embedding dimension
num_heads = 4  # Number of attention heads
num_layers = 2  # Number of transformer layers
dropout = 0.1  # Dropout rate

#graph_colomns=5
num_components=5
batch_size = 16  # Batch size

graph_input_dim = 310  # Number of colomns in the graph
text_vocab_size = 894  # Vocabulary size for text


In [56]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

### Data Load

In [57]:
num_ciruits = 320
graph_dataset =  circuits.graphs
text_dataset = circuits.component_indices

# Convert graph_dataset and text_dataset to numpy arrays
graph_dataset = np.array(graph_dataset, dtype=object)
text_dataset = np.array(text_dataset, dtype=object)

# Convert numpy arrays to PyTorch tensors
graph_dataset = [torch.tensor(graph, dtype=torch.float32) for graph in graph_dataset]
graph_dataset = [torch.cat((torch.nn.functional.pad(graph, (0, max_length - graph.size(0))),torch.tensor([[9]*310]))) for graph in graph_dataset]
text_dataset = [torch.tensor(text+[893], dtype=torch.int) for text in text_dataset]



In [58]:
# Combine graph_dataset and text_dataset into a single list of tuples
combined_dataset = list(zip(graph_dataset, text_dataset))

# Shuffle the combined dataset
np.random.shuffle(combined_dataset)

# Unzip the shuffled dataset back into graph_dataset and text_dataset
graph_dataset, text_dataset = zip(*combined_dataset)

# Convert back to the original data types
graph_dataset = list(graph_dataset)
text_dataset = list(text_dataset)

In [59]:
graph_dataset = torch.cat(graph_dataset).to(device)
text_dataset = torch.cat(text_dataset,).to(device)

In [60]:
graph_train = graph_dataset[:int(0.8 * len(graph_dataset))]
graph_val = graph_dataset[int(0.8 * len(graph_dataset)):]
text_train = text_dataset[:int(0.8 * len(text_dataset))]
text_val = text_dataset[int(0.8 * len(text_dataset)):]

### Generate Batchers

In [61]:
batch_size = 16
block_size = 16


def get_batch( batch_size=4, block_size=16, train=True):

    if train:
        graph_dataset = graph_train
        text_dataset = text_train
    else:
        graph_dataset = graph_val
        text_dataset = text_val
    
    start_indices = torch.randint(0, len(graph_dataset) - block_size, (batch_size,))
    """
    Get a batch of sequences from the text_dataset.

    Args:
        text_dataset (torch.Tensor): The dataset containing text sequences.
        batch_size (int): The number of sequences in the batch.
        block_size (int): The length of each sequence block.

    Returns:
        torch.Tensor: A batch of sequences with shape (batch_size, block_size).
    """
    # Combine graph data and text data for the batch
    
    batch_graph = torch.stack([graph_dataset[i:i + block_size] for i in start_indices])
    batch_text = torch.stack([text_dataset[i:i + block_size] for i in start_indices])
    # Extract sequences of length block_size starting from the sampled indices

    return batch_text, batch_graph 




### Importing the model

In [62]:
model = GtoTmodel.GraphToTextTransformer(
    graph_input_dim, 
    text_vocab_size, 
    embed_dim, 
    num_heads, 
    num_layers, 
    dropout)

model = model.to(device)
batch_text, batch_graph = get_batch( batch_size=4, block_size=16)
batch_text = batch_text.to(device)
batch_graph = batch_graph.to(device)


/home/nithira/circuits_gen/.venv/lib/python3.10/site-packages/torch/nn/modules/transformer.py:385: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.self_attn.batch_first was not True(use batch_first for better inference performance)
  warnings.warn(


In [63]:
learning_rate = 0.001
num_epochs = 10

# Define loss function and optimizer
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=learning_rate)

### Training

In [67]:
# model.to('cpu')
# atch_text = batch_text.to('cpu')
# batch_graph = batch_graph.to('cpu')
batch_text, batch_graph = get_batch( batch_size=4, block_size=16)
for _ in range(10000):
    batch_text, batch_graph = get_batch(batch_size=4, block_size=16)

    # Forward pass through the model
    # Ensure batch_text has at least two dimensions
    if batch_text.dim() == 1:
        batch_text = batch_text.unsqueeze(0)  # Add a batch dimension if missing

    # Ensure batch_graph has at least two dimensions
    if batch_graph.dim() == 1:
        batch_graph = batch_graph.unsqueeze(0)  # Add a batch dimension if missing

    # Forward pass through the model
    output = model(batch_graph, batch_text[:, :-1])  # Exclude the last token for input

    print("Output shape:", output.shape)  # Should be (batch_size, seq_length, vocab_size)

    # Calculate loss
    loss = criterion(output.reshape(-1, text_vocab_size), batch_text[:, 1:].reshape(-1).long())  # Exclude the first token for target
    print("Loss:", loss.item())  # Print the loss value
    # Backward pass and optimization
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    # Print the updated model parameter

Output shape: torch.Size([4, 15, 894])
Loss: 0.1118939146399498
Output shape: torch.Size([4, 15, 894])
Loss: 0.029046697542071342
Output shape: torch.Size([4, 15, 894])
Loss: 0.2156154364347458
Output shape: torch.Size([4, 15, 894])
Loss: 0.11459264904260635
Output shape: torch.Size([4, 15, 894])
Loss: 0.07923902571201324
Output shape: torch.Size([4, 15, 894])
Loss: 0.110430046916008
Output shape: torch.Size([4, 15, 894])
Loss: 0.11908137053251266
Output shape: torch.Size([4, 15, 894])
Loss: 0.13190481066703796
Output shape: torch.Size([4, 15, 894])
Loss: 0.06236695498228073
Output shape: torch.Size([4, 15, 894])
Loss: 0.11451864242553711
Output shape: torch.Size([4, 15, 894])
Loss: 0.1647203415632248
Output shape: torch.Size([4, 15, 894])
Loss: 0.16431449353694916
Output shape: torch.Size([4, 15, 894])
Loss: 0.42103904485702515
Output shape: torch.Size([4, 15, 894])
Loss: 0.05565702170133591
Output shape: torch.Size([4, 15, 894])
Loss: 0.4082771837711334
Output shape: torch.Size([4, 1

### Validation loop

In [68]:
model.eval()  # Set the model to evaluation mode
total_val_loss = 0

with torch.no_grad():  # Disable gradient computation for validation
    for _ in range(1000):  # Number of validation iterations
        batch_text, batch_graph = get_batch(batch_size=4, block_size=16, train=False)

        # Ensure batch_text has at least two dimensions
        if batch_text.dim() == 1:
            batch_text = batch_text.unsqueeze(0)  # Add a batch dimension if missing

        # Ensure batch_graph has at least two dimensions
        if batch_graph.dim() == 1:
            batch_graph = batch_graph.unsqueeze(0)  # Add a batch dimension if missing

        # Forward pass through the model
        output = model(batch_graph, batch_text[:, :-1])  # Exclude the last token for input

        # Calculate loss
        loss = criterion(output.reshape(-1, text_vocab_size), batch_text[:, 1:].reshape(-1).long())  # Exclude the first token for target
        total_val_loss += loss.item()

        # Print validation loss
        print("Validation Loss:", loss.item())

# Calculate average validation loss
average_val_loss = total_val_loss / 1000
# Print the average validation loss for the current epoch
print(f"Epoch {_ + 1}, Average Validation Loss: {average_val_loss}")

Validation Loss: 0.08359275013208389
Validation Loss: 0.14742828905582428
Validation Loss: 0.025035589933395386
Validation Loss: 0.08904464542865753
Validation Loss: 0.04201240465044975
Validation Loss: 0.037293750792741776
Validation Loss: 0.030215386301279068
Validation Loss: 0.0033351904712617397
Validation Loss: 0.021105937659740448
Validation Loss: 0.0037451342213898897
Validation Loss: 0.02126489207148552
Validation Loss: 0.11054427921772003
Validation Loss: 0.19875553250312805
Validation Loss: 0.2316560596227646
Validation Loss: 0.11250178515911102
Validation Loss: 0.025432778522372246
Validation Loss: 0.01985360123217106
Validation Loss: 0.0580269955098629
Validation Loss: 0.056294072419404984
Validation Loss: 0.07841923832893372
Validation Loss: 0.02282976172864437
Validation Loss: 0.0869392529129982
Validation Loss: 0.04787830635905266
Validation Loss: 0.08865243941545486
Validation Loss: 0.07923515886068344
Validation Loss: 0.11941300332546234
Validation Loss: 0.086216256022

In [66]:
# # Get a batch of data


# # Move the batch to the appropriate device

# batch_text, batch_graph = get_batch(batch_size=4, block_size=16)
# batch_text = batch_text.to(device)
# batch_graph = batch_graph.to(device)
# # Forward pass through the model
# # Ensure batch_text has at least two dimensions
# if batch_text.dim() == 1:
#     batch_text = batch_text.unsqueeze(0)  # Add a batch dimension if missing

# # Ensure batch_graph has at least two dimensions
# if batch_graph.dim() == 1:
#     batch_graph = batch_graph.unsqueeze(0)  # Add a batch dimension if missing

# # Forward pass through the model
# output = model(batch_graph, batch_text[:, :-1])  # Exclude the last token for input

# print("Output shape:", output.shape)  # Should be (batch_size, seq_length, vocab_size)

# # Calculate loss
# loss = criterion(output.reshape(-1, text_vocab_size), batch_text[:, 1:].reshape(-1).long())  # Exclude the first token for target
# print("Loss:", loss.item())  # Print the loss value
# # Backward pass and optimization
# optimizer.zero_grad()
# loss.backward()
# optimizer.step()
# # Print the updated model parameters
# for name, param in model.named_parameters():
#     if param.requires_grad:
#         print(name, param.data)